# Notebook 05: Classification and Clustering

Demonstrates KNN, SVM, and Ward hierarchical clustering on adsorption descriptor data using scikit-learn.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

## Load descriptor dataset

Place your CSV/Excel file in the same folder as this notebook.  
Required: at least 2 numeric descriptor columns + 1 **group** column (classification target).  
Optional categorical columns (e.g. `isotherm_type`, `material_type`) are one-hot encoded automatically.

Three sample files are provided:
| File | Columns |
|------|---------|
| `sample_descriptors_basic.csv` | Q_max, Eads, sigma_E, **group** |
| `sample_descriptors_isotherm.csv` | + isotherm_type (categorical) |
| `sample_descriptors_full.csv` | + isotherm_type + material_type (categoricals) |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        DATA FILE CONFIGURATION                              ║
# ║                                                                              ║
# ║  Set DATA_FILE to the name of your descriptor CSV/Excel file.               ║
# ║  Place the file in the SAME FOLDER as this notebook.                        ║
# ║                                                                              ║
# ║  The file must contain:                                                      ║
# ║    - Numeric columns  : Q_max, Eads, sigma_E, etc. (descriptors)            ║
# ║    - A 'group' column : classification target (e.g. High/Medium/Low)        ║
# ║    - Optional categoricals: isotherm_type, material_type, etc.              ║
# ║      (these are one-hot encoded automatically)                               ║
# ║                                                                              ║
# ║  Sample files included for testing:                                          ║
# ║    "sample_descriptors_basic.csv"      (numeric only)                       ║
# ║    "sample_descriptors_isotherm.csv"   (+ isotherm_type)                    ║
# ║    "sample_descriptors_full.csv"       (+ isotherm_type + material_type)    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

DATA_FILE  = "sample_descriptors_full.csv"   # <--- CHANGE THIS to your file
TARGET_COL = "group"                          # <--- column with class labels

# ══════════════════════════════════════════════════════════════════════════════
#  Do NOT modify anything below this line unless you know what you are doing.
# ══════════════════════════════════════════════════════════════════════════════

# ── Resolve path ──────────────────────────────────────────────────────────────
_NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
DATA_PATH = os.path.join(_NOTEBOOK_DIR, DATA_FILE)

if not DATA_FILE or DATA_FILE.strip() == "":
    raise ValueError(
        "\n\n  *** ERROR: No data file specified! ***\n\n"
        "  Set DATA_FILE at the top of this cell.\n"
        "  Example:  DATA_FILE = 'sample_descriptors_basic.csv'\n"
    )
if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(
        f"\n\n  *** ERROR: File not found: '{DATA_FILE}' ***\n\n"
        f"  Looked in: {_NOTEBOOK_DIR}\n"
        f"  Place your file in the same folder as this notebook.\n"
    )

# ── Load ──────────────────────────────────────────────────────────────────────
ext = os.path.splitext(DATA_FILE)[1].lower()
if ext == '.csv':
    df_raw = pd.read_csv(DATA_PATH)
elif ext in ('.xlsx', '.xls'):
    df_raw = pd.read_excel(DATA_PATH)
else:
    raise ValueError(f"Unsupported format '{ext}'. Use .csv or .xlsx")

# ── Validate target column ────────────────────────────────────────────────────
if TARGET_COL not in df_raw.columns:
    raise ValueError(
        f"\n\n  *** ERROR: Target column '{TARGET_COL}' not found! ***\n\n"
        f"  Your file has columns: {list(df_raw.columns)}\n"
        f"  Set TARGET_COL to the column that contains group labels.\n"
    )

# ── Separate target, numeric features, and categorical features ───────────────
y_labels = df_raw[TARGET_COL].values
label_enc = LabelEncoder()
y = label_enc.fit_transform(y_labels)
group_names = list(label_enc.classes_)

feature_cols = [c for c in df_raw.columns if c != TARGET_COL]
df_feat = df_raw[feature_cols].copy()

numeric_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_feat.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Loaded {len(df_raw)} samples from '{DATA_FILE}'")
print(f"  Target column : '{TARGET_COL}'  ->  {len(group_names)} classes: {group_names}")
print(f"  Numeric features  ({len(numeric_cols)}): {numeric_cols}")
print(f"  Categorical features ({len(cat_cols)}): {cat_cols if cat_cols else '(none)'}")

# ── One-hot encode categoricals ───────────────────────────────────────────────
if cat_cols:
    df_onehot = pd.get_dummies(df_feat, columns=cat_cols, drop_first=False, dtype=float)
    print(f"\n  After one-hot encoding: {df_onehot.shape[1]} features")
    print(f"    {list(df_onehot.columns)}")
else:
    df_onehot = df_feat.copy()

X = df_onehot.values.astype(float)
feature_names = list(df_onehot.columns)

# ── Scale ─────────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\n  Feature matrix shape: {X_scaled.shape}  (samples x features)")
df_raw.head()

## KNN classification with leave-one-out cross-validation

In [ ]:
# KNN with leave-one-out cross-validation (all features)
knn = KNeighborsClassifier(n_neighbors=min(5, len(y) - 1))
loo = LeaveOneOut()
scores = cross_val_score(knn, X_scaled, y, cv=loo)
print(f'KNN (k={knn.n_neighbors}) LOO accuracy: {scores.mean():.2%}')
print(f'  Correct: {int(scores.sum())}/{len(scores)}')

# Fit on first 2 numeric features for 2D visualisation
idx_x, idx_y = 0, 1  # Q_max and Eads (first two numeric cols)
X_2d = X_scaled[:, [idx_x, idx_y]]
knn_2d = KNeighborsClassifier(n_neighbors=knn.n_neighbors)
knn_2d.fit(X_2d, y)

# Decision region
fig, ax = plt.subplots(figsize=(8, 6))
x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                      np.linspace(y_min, y_max, 200))
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

import matplotlib.cm as cm
_colors = [cm.Set2(i / max(len(group_names)-1, 1)) for i in range(len(group_names))]

ax.contourf(xx, yy, Z, alpha=0.2, levels=np.arange(-0.5, len(group_names)), cmap='Set2')
for i, name in enumerate(group_names):
    mask = y == i
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[_colors[i]], label=name, edgecolors='k', s=60)

ax.set_xlabel(f'{numeric_cols[0]} (standardised)', fontsize=11)
ax.set_ylabel(f'{numeric_cols[1]} (standardised)', fontsize=11)
ax.set_title(f'KNN Decision Regions (k={knn.n_neighbors}, 2D projection)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

# Full-feature classification report
knn.fit(X_scaled, y)
y_pred_knn = knn.predict(X_scaled)
print('\nClassification report (full feature set, resubstitution):')
print(classification_report(y, y_pred_knn, target_names=group_names))

## SVM classification

In [ ]:
# SVM classification (all features, linear kernel)
n_folds = min(5, min(np.bincount(y)))  # safe number of folds
svm = SVC(kernel='linear', C=1.0)
svm_scores = cross_val_score(svm, X_scaled, y, cv=n_folds)
print(f'SVM (linear) {n_folds}-fold CV accuracy: {svm_scores.mean():.2%}')

# 2D visualisation on first two numeric features
svm_2d = SVC(kernel='linear', C=1.0)
svm_2d.fit(X_2d, y)

fig, ax = plt.subplots(figsize=(8, 6))
Z_svm = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z_svm, alpha=0.2, levels=np.arange(-0.5, len(group_names)), cmap='Set2')
ax.contour(xx, yy, Z_svm, colors='k', linewidths=0.5)

for i, name in enumerate(group_names):
    mask = y == i
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[_colors[i]], label=name, edgecolors='k', s=60)

# Support vectors
ax.scatter(svm_2d.support_vectors_[:, 0], svm_2d.support_vectors_[:, 1],
           s=150, facecolors='none', edgecolors='red', linewidths=2,
           label='Support vectors')

ax.set_xlabel(f'{numeric_cols[0]} (standardised)', fontsize=11)
ax.set_ylabel(f'{numeric_cols[1]} (standardised)', fontsize=11)
ax.set_title('SVM Decision Boundary (linear kernel, 2D projection)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

# Full-feature classification report
svm_full = SVC(kernel='linear', C=1.0)
svm_full.fit(X_scaled, y)
y_pred_svm = svm_full.predict(X_scaled)
print('\nClassification report (full feature set, resubstitution):')
print(classification_report(y, y_pred_svm, target_names=group_names))

## Ward hierarchical clustering

In [ ]:
# Ward hierarchical clustering
Z_link = linkage(X_scaled, method='ward')

fig, ax = plt.subplots(figsize=(12, 5))
sample_labels = [f'{group_names[yi]}-{i}' for i, yi in enumerate(y)]
dendrogram(Z_link, ax=ax, color_threshold=5.0, labels=sample_labels)
ax.axhline(y=5.0, color='r', linestyle='--', label='Cut threshold')
ax.set_ylabel('Ward linkage distance', fontsize=11)
ax.set_title('Hierarchical Clustering Dendrogram', fontsize=13)
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

## Cluster validation

In [ ]:
cluster_labels = fcluster(Z_link, t=len(group_names), criterion='maxclust')

print('Confusion matrix (true groups vs clusters):')
print(confusion_matrix(y, cluster_labels - 1))
print(f'\nAdjusted Rand Index: {adjusted_rand_score(y, cluster_labels):.3f}')

# 3D scatter on first 3 numeric features
n0, n1, n2 = numeric_cols[0], numeric_cols[1], numeric_cols[min(2, len(numeric_cols)-1)]
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

for i, name in enumerate(group_names):
    mask = y == i
    ax.scatter(df_raw.loc[mask, n0], df_raw.loc[mask, n1], df_raw.loc[mask, n2],
               c=[_colors[i]], label=name, s=60, edgecolors='k')

ax.set_xlabel(n0, fontsize=10)
ax.set_ylabel(n1, fontsize=10)
ax.set_zlabel(n2, fontsize=10)
ax.set_title('3D Descriptor Space', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

# Show feature importance summary
if cat_cols:
    print(f"\nNote: {len(cat_cols)} categorical feature(s) were one-hot encoded:")
    for c in cat_cols:
        vals = df_raw[c].unique()
        print(f"  {c}: {list(vals)}")
    print(f"  Total features used for classification: {X_scaled.shape[1]}")